## MLP Model Traaining

In [ ]:
import os
import numpy as np
import torch
import torch.nn as nn
from torch.optim import Adam
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler
from sklearn.metrics import (
    f1_score,
    recall_score,
    precision_score,
    accuracy_score,
    classification_report,
    confusion_matrix,
)
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight


# ============================================================
# Dataset Class
# ============================================================

class HiddenData(Dataset):
    def __init__(self, data: torch.Tensor, labels: torch.Tensor):
        self.data = data
        self.labels = labels

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        return self.data[idx], self.labels[idx]


# ============================================================
# MLP Model
# ============================================================

class MLPNet(nn.Module):
    def __init__(self, in_dim=4096, dropout=0.3, out_dim=2):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(in_dim, 512),
            nn.ReLU(),
            nn.Linear(512, 512),
            nn.ReLU(),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Linear(256, 256),
            nn.ReLU(),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Linear(128, 128),
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, out_dim),
        )

    def forward(self, x):
        return self.layers(x)


# ============================================================
# Utility
# ============================================================

def prepare_hidden_states(hidden_states: torch.Tensor) -> torch.Tensor:
    """
    Expected final shape: [N, D]
    Handles:
    - [N, D]
    - [N, 1, D] -> squeeze to [N, D]
    """
    if not isinstance(hidden_states, torch.Tensor):
        hidden_states = torch.tensor(hidden_states, dtype=torch.float32)

    hidden_states = hidden_states.float()

    if hidden_states.dim() == 3 and hidden_states.shape[1] == 1:
        hidden_states = hidden_states.squeeze(1)

    if hidden_states.dim() != 2:
        raise ValueError(
            f"hidden_states must have shape [N, D] or [N, 1, D], got {tuple(hidden_states.shape)}"
        )

    return hidden_states


def prepare_labels(labels: torch.Tensor) -> torch.Tensor:
    if not isinstance(labels, torch.Tensor):
        labels = torch.tensor(labels, dtype=torch.long)
    return labels.long().view(-1)


def evaluate_model(model, data_loader, device):
    model.eval()
    y_true, y_pred = [], []

    with torch.no_grad():
        for h, l in data_loader:
            h = h.to(device, dtype=torch.float32)
            l = l.to(device, dtype=torch.long)

            out = model(h)
            pred = torch.argmax(out, dim=1)

            y_true.extend(l.cpu().numpy())
            y_pred.extend(pred.cpu().numpy())

    metrics = {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision_macro": precision_score(y_true, y_pred, average="macro", zero_division=0),
        "recall_macro": recall_score(y_true, y_pred, average="macro", zero_division=0),
        "f1_macro": f1_score(y_true, y_pred, average="macro", zero_division=0),
    }

    return metrics, y_true, y_pred


# ============================================================
# Training Function
# ============================================================

def train_mlp(
    hidden_states: torch.Tensor,
    labels: torch.Tensor,
    output_dir: str,
    dropout: float = 0.5,
    batch_size: int = 32,
    epochs: int = 100,
    lr_rate: float = 1e-4,
):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    os.makedirs(output_dir, exist_ok=True)
    weight_dir = os.path.join(output_dir, "weights")
    os.makedirs(weight_dir, exist_ok=True)

    hidden_states = prepare_hidden_states(hidden_states)
    labels = prepare_labels(labels)

    if len(hidden_states) != len(labels):
        raise ValueError(
            f"Number of hidden states ({len(hidden_states)}) != number of labels ({len(labels)})"
        )

    labels_np = labels.cpu().numpy()
    unique_classes = np.unique(labels_np)

    if len(unique_classes) < 2:
        raise ValueError("Need at least 2 classes in labels to train the MLP.")

    # Stratified split: 80 / 10 / 10
    X_temp, X_test, y_temp, y_test = train_test_split(
        hidden_states,
        labels,
        test_size=0.1,
        random_state=42,
        stratify=labels_np,
    )

    X_train, X_dev, y_train, y_dev = train_test_split(
        X_temp,
        y_temp,
        test_size=0.1111,  # ~10% of total
        random_state=42,
        stratify=y_temp.cpu().numpy(),
    )
    # Class weights
    y_train_np = y_train.cpu().numpy()
    class_weights = compute_class_weight(
        class_weight="balanced",
        classes=np.unique(y_train_np),
        y=y_train_np,
    )
    class_weights_tensor = torch.tensor(class_weights, dtype=torch.float32).to(device)

    # Weighted sampler
    sample_weights = np.array(
        [class_weights[int(label.item())] for label in y_train],
        dtype=np.float32
    )

    sampler = WeightedRandomSampler(
        weights=torch.tensor(sample_weights, dtype=torch.float32),
        num_samples=len(sample_weights),
        replacement=True,
    )

    # DataLoaders
    train_dataset = HiddenData(X_train, y_train)
    dev_dataset = HiddenData(X_dev, y_dev)
    test_dataset = HiddenData(X_test, y_test)

    train_loader = DataLoader(train_dataset, batch_size=batch_size, sampler=sampler)
    dev_loader = DataLoader(dev_dataset, batch_size=batch_size, shuffle=False)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

    # Model
    input_dim = hidden_states.shape[1]
    model = MLPNet(in_dim=input_dim, dropout=dropout, out_dim=2).to(device)

    optimizer = Adam(model.parameters(), lr=lr_rate)
    criterion = nn.CrossEntropyLoss(weight=class_weights_tensor)

    best_dev_f1 = -1.0
    best_model_path = os.path.join(weight_dir, "best_model.pth")

    for epoch in range(epochs):
        model.train()
        total_loss = 0.0

        for h, l in train_loader:
            h = h.to(device, dtype=torch.float32)
            l = l.to(device, dtype=torch.long)

            optimizer.zero_grad()
            out = model(h)
            loss = criterion(out, l)
            loss.backward()
            optimizer.step()

            total_loss += loss.item()

        avg_loss = total_loss / max(len(train_loader), 1)

        # Dev evaluation
        dev_metrics, _, _ = evaluate_model(model, dev_loader, device)
        dev_f1 = dev_metrics["f1_macro"]

        if dev_f1 > best_dev_f1:
            best_dev_f1 = dev_f1
            torch.save(model.state_dict(), best_model_path)

        if (epoch + 1) % 5 == 0 or epoch == 0:
            print(
                f"Epoch {epoch + 1}/{epochs} | "
                f"Loss={avg_loss:.4f} | "
            )

    # Final Test evaluation
    model.load_state_dict(torch.load(best_model_path, map_location=device))
    test_metrics, y_test_true, y_test_pred = evaluate_model(model, test_loader, device)

    print("\n================ TEST RESULTS ================")
    print(f"Accuracy : {test_metrics['accuracy']:.4f}") 
    print(f"Precision (Macro) : {test_metrics['precision_macro']:.4f}") 
    print(f"Recall (Macro) : {test_metrics['recall_macro']:.4f}") 
    print(f"F1 Score (Macro) : {test_metrics['f1_macro']:.4f}") 

    return best_model_path


# ============================================================
# MAIN
# ============================================================

if __name__ == "__main__":
    OUTPUT_DIR = "./training_data_mlp"
    os.makedirs(OUTPUT_DIR, exist_ok=True)

    # First dataset
    hidden_states_path_1 = os.path.join(OUTPUT_DIR, "hidden_states.pt")
    labels_path_1 = os.path.join(OUTPUT_DIR, "labels.pt")

    # Second dataset
    hidden_states_path_2 = os.path.join(OUTPUT_DIR, "training_data_mlp_test\\hidden_states.pt")
    labels_path_2 = os.path.join(OUTPUT_DIR, "training_data_mlp_test\\labels.pt")

    for path in [hidden_states_path_1, labels_path_1, hidden_states_path_2, labels_path_2]:
        if not os.path.exists(path):
            raise FileNotFoundError(f"File not found: {path}")

    # Load first dataset
    hidden_states_1 = torch.load(hidden_states_path_1, map_location="cpu")
    labels_1 = torch.load(labels_path_1, map_location="cpu")

    # Load second dataset
    hidden_states_2 = torch.load(hidden_states_path_2, map_location="cpu")
    labels_2 = torch.load(labels_path_2, map_location="cpu")

    # Prepare
    hidden_states_1 = prepare_hidden_states(hidden_states_1)
    labels_1 = prepare_labels(labels_1)

    hidden_states_2 = prepare_hidden_states(hidden_states_2)
    labels_2 = prepare_labels(labels_2)

    # Check feature dimension matches
    if hidden_states_1.shape[1] != hidden_states_2.shape[1]:
        raise ValueError(
            f"Feature dimensions do not match: "
            f"{hidden_states_1.shape[1]} vs {hidden_states_2.shape[1]}"
        )

    # Combine datasets
    hidden_states = torch.cat([hidden_states_1, hidden_states_2], dim=0)
    labels = torch.cat([labels_1, labels_2], dim=0)

    best_model_path = train_mlp(
        hidden_states=hidden_states,
        labels=labels,
        output_dir=OUTPUT_DIR,
        dropout=0.5,
        batch_size=32,
        epochs=100,
        lr_rate=1e-4,
    )

    print(f"\nTraining complete. Best model path: {best_model_path}")

Epoch 1/100 | Loss=0.5389 | 
Epoch 5/100 | Loss=0.4284 | 
Epoch 10/100 | Loss=0.3553 | 
Epoch 15/100 | Loss=0.2591 | 
Epoch 20/100 | Loss=0.1911 | 
Epoch 25/100 | Loss=0.1208 | 
Epoch 30/100 | Loss=0.0937 | 
Epoch 35/100 | Loss=0.0770 | 
Epoch 40/100 | Loss=0.0563 | 
Epoch 45/100 | Loss=0.0544 | 
Epoch 50/100 | Loss=0.0503 | 
Epoch 55/100 | Loss=0.0461 | 
Epoch 60/100 | Loss=0.0467 | 
Epoch 65/100 | Loss=0.0371 | 
Epoch 70/100 | Loss=0.0424 | 
Epoch 75/100 | Loss=0.0198 | 
Epoch 80/100 | Loss=0.0175 | 
Epoch 85/100 | Loss=0.0253 | 
Epoch 90/100 | Loss=0.0181 | 
Epoch 95/100 | Loss=0.0141 | 
Epoch 100/100 | Loss=0.0251 | 

================ TEST RESULTS ================
Accuracy          : 0.8124
Precision (Macro) : 0.7851
Recall (Macro)    : 0.8013
F1 Score (Macro)  : 0.7928

Training complete. Best model path: ./training_data_mlp\weights\best_model.pth


## BERT reranker training from scratch

In [ ]:
import os
import torch

from src.reranker import Reranker, RerankerTrainer
from src.reranker.data import GroupedTrainDataset, PredictionDataset, GroupCollator
from transformers import AutoTokenizer, AutoConfig, TrainingArguments

# -------------------
# Model
# -------------------
model_name = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)
dev_path = "biencoder-nq-dev-training-data2"

# -------------------
# Data args
# -------------------
class DataArgs:
    # training data (grouped format)
    train_path = "/content/drive/MyDrive/Data/biencoder-nq-dev-training-data.json"
    dev_path = "/content/drive/MyDrive/biencoder-nq-dev-training-data2.json"
    train_group_size = 8          
    max_len = 512                 


class ModelArgs:
    untie_encoder = False
    temperature = 1.0

data_args = DataArgs()
model_args = ModelArgs()

# -------------------
# Training args
# -------------------
training_args = TrainingArguments(
    output_dir="/content/drive/MyDrive/bert_reranker_ckpt_2",
    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,   
    gradient_accumulation_steps=4,
    learning_rate=2e-5,
    num_train_epochs=3,
    logging_steps=50,
    save_steps=1000,
    fp16=True,
    dataloader_num_workers=2,
    remove_unused_columns=False,
    warmup_ratio=0.1,
    warmup_steps=0
)

training_args.collaborative = False
training_args.distance_cache = False
training_args.local_rank = -1

# -------------------
# Optional config
# -------------------
config = AutoConfig.from_pretrained(
    model_name,
    num_labels=1,
)

# -------------------
# Load model
# -------------------
model = Reranker.from_pretrained(
    model_args,
    data_args,
    training_args,
    model_name,
    config=config,
)

# -------------------
# Train dataset
# -------------------
train_dataset = GroupedTrainDataset(
    data_args,
    data_args.train_path,
    tokenizer=tokenizer,
    train_args=training_args
)
eval_dataset = GroupedTrainDataset(
        data_args,
        data_args.dev_path,
        tokenizer=tokenizer,
        train_args=training_args
    )

# -------------------
# Trainer
# -------------------
trainer = RerankerTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    data_collator=GroupCollator(tokenizer),
)
trainer.tokenizer = tokenizer

# -------------------
# Train
# -------------------
trainer.train()

# -------------------
# Save model + tokenizer
# -------------------
trainer.save_model()
tokenizer.save_pretrained(training_args.output_dir)

# -------------------
# Prediction
# -------------------


warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Step,Training Loss
50,1.757142
100,1.316751
150,0.472073
200,0.074886
250,0.030499
300,0.018184
350,0.010372
400,0.009722
450,0.004776
500,0.003533


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('/content/drive/MyDrive/bert_reranker_ckpt_2/tokenizer_config.json',
 '/content/drive/MyDrive/bert_reranker_ckpt_2/tokenizer.json')

In [6]:
import torch
from transformers import AutoTokenizer
from src.reranker import RerankerForInference
from src.reranker.data import PredictionDataset
from torch.utils.data import DataLoader

model_path = "/content/drive/MyDrive/bert_reranker_ckpt_2"

model = RerankerForInference.from_pretrained(model_path)
tokenizer = model.tokenizer

device = "cuda"
model.hf_model.to(device)
model.hf_model.eval()

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12,

## BERT Reranker(Trained from scratch)  evaluation

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import json
from collections import defaultdict

import torch
from sentence_transformers import CrossEncoder

# =========================
# CONFIG
# =========================
TEST_FILE = "/content/drive/MyDrive/Data/biencoder-nq-dev-training-data.json"
MODEL_PATH = "/content/drive/MyDrive/bert_reranker_ckpt_2"

BATCH_SIZE = 32
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print("Using device:", DEVICE)
print("Model:", MODEL_PATH)

# =========================
# LOAD TEST SET
# =========================
with open(TEST_FILE, "r", encoding="utf-8") as f:
    data = json.load(f)

pairs = []
qrels = defaultdict(dict)

for item in data:
    qid = str(item["qry"]["qid"])
    query = item["qry"]["query"]

    for p in item.get("pos", []):
        pid = str(p["pid"])
        passage = p["passage"]
        pairs.append((qid, pid, query, passage))
        qrels[qid][pid] = 1

    for n in item.get("neg", []):
        pid = str(n["pid"])
        passage = n["passage"]
        pairs.append((qid, pid, query, passage))
        if pid not in qrels[qid]:
            qrels[qid][pid] = 0

print("Total query-passage pairs:", len(pairs))
print("Total queries:", len(qrels))

# =========================
# LOAD MODEL
# =========================
model = CrossEncoder(MODEL_PATH, device=DEVICE)

model_inputs = [[query, passage] for _, _, query, passage in pairs]

scores = model.predict(
    model_inputs,
    batch_size=BATCH_SIZE,
    show_progress_bar=True
)

run = defaultdict(dict)
for (qid, pid, _, _), score in zip(pairs, scores):
    run[qid][pid] = float(score)

# =========================
# METRIC FUNCTIONS
# =========================
def precision_at_k(sorted_pids, rels, k):
    top_k = sorted_pids[:k]
    if len(top_k) == 0:
        return 0.0
    relevant_in_top_k = sum(1 for pid in top_k if rels.get(pid, 0) > 0)
    return relevant_in_top_k / k

def recall_at_k(sorted_pids, rels, k):
    relevant = {pid for pid, rel in rels.items() if rel > 0}
    if not relevant:
        return 0.0
    retrieved = set(sorted_pids[:k])
    return len(relevant & retrieved) / len(relevant)

def mrr_at_k(sorted_pids, rels, k):
    for i, pid in enumerate(sorted_pids[:k], start=1):
        if rels.get(pid, 0) > 0:
            return 1.0 / i
    return 0.0

# =========================
# COMPUTE METRICS
# =========================
p1_scores, p3_scores, p5_scores = [], [], []
r1_scores, r3_scores, r5_scores = [], [], []
mrr1_scores, mrr3_scores, mrr5_scores = [], [], []

for qid, rels in qrels.items():
    if qid not in run:
        continue

    ranked = sorted(run[qid].items(), key=lambda x: x[1], reverse=True)
    sorted_pids = [pid for pid, _ in ranked]

    p1_scores.append(precision_at_k(sorted_pids, rels, 1))
    p3_scores.append(precision_at_k(sorted_pids, rels, 3))
    p5_scores.append(precision_at_k(sorted_pids, rels, 5))

    r1_scores.append(recall_at_k(sorted_pids, rels, 1))
    r3_scores.append(recall_at_k(sorted_pids, rels, 3))
    r5_scores.append(recall_at_k(sorted_pids, rels, 5))

    mrr1_scores.append(mrr_at_k(sorted_pids, rels, 1))
    mrr3_scores.append(mrr_at_k(sorted_pids, rels, 3))
    mrr5_scores.append(mrr_at_k(sorted_pids, rels, 5))

results = {
    "Precision@1": sum(p1_scores) / len(p1_scores) if p1_scores else 0.0,
    "Precision@3": sum(p3_scores) / len(p3_scores) if p3_scores else 0.0,
    "Precision@5": sum(p5_scores) / len(p5_scores) if p5_scores else 0.0,

    "Recall@1": sum(r1_scores) / len(r1_scores) if r1_scores else 0.0,
    "Recall@3": sum(r3_scores) / len(r3_scores) if r3_scores else 0.0,
    "Recall@5": sum(r5_scores) / len(r5_scores) if r5_scores else 0.0,

    "MRR@1": sum(mrr1_scores) / len(mrr1_scores) if mrr1_scores else 0.0,
    "MRR@3": sum(mrr3_scores) / len(mrr3_scores) if mrr3_scores else 0.0,
    "MRR@5": sum(mrr5_scores) / len(mrr5_scores) if mrr5_scores else 0.0,
}

print("\nEvaluation Results for BERT Reranker(Trained from scratch):")
for k, v in results.items():
    print(f"{k}: {v:.4f}")

Using device: cuda
Model: /content/drive/MyDrive/bert_reranker_ckpt_2
Total query-passage pairs: 59705
Total queries: 5926


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Batches:   0%|          | 0/1866 [00:00<?, ?it/s]


Evaluation Results for BERT Reranker(Trained from scratch):
Precision@1: 0.9238
Precision@3: 0.7532
Precision@5: 0.6621
Recall@1: 0.4523
Recall@3: 0.7627
Recall@5: 0.8914
MRR@1: 0.9067
MRR@3: 0.9285
MRR@5: 0.9412


#finetuning

In [ ]:
import json
import math
import os

import torch
from torch.utils.data import DataLoader

from sentence_transformers.cross_encoder import CrossEncoder
from sentence_transformers.readers import InputExample

# =========================
# CONFIG
# =========================
TRAIN_FILE = "/content/drive/MyDrive/Data/nq_reranker_raw.json"
MODEL_NAME = "cross-encoder/ms-marco-TinyBERT-L2-v2"
OUTPUT_DIR = "/content/drive/MyDrive/tinybert_msmarco_finetuned"

MAX_LENGTH = 512
TRAIN_BATCH_SIZE = 16
NUM_EPOCHS = 3
LEARNING_RATE = 2e-5
WARMUP_RATIO = 0.1

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", DEVICE)
print("Model:", MODEL_NAME)

# =========================
# LOAD GROUPED DATA
# =========================
with open(TRAIN_FILE, "r", encoding="utf-8") as f:
    train_grouped = json.load(f)

# =========================
# CONVERT TO PAIRWISE TRAINING DATA
# label 1.0 = positive
# label 0.0 = negative
# =========================
train_examples = []

for item in train_grouped:
    query = item["qry"]["query"]

    for p in item.get("pos", []):
        train_examples.append(
            InputExample(texts=[query, p["passage"]], label=1.0)
        )

    for n in item.get("neg", []):
        train_examples.append(
            InputExample(texts=[query, n["passage"]], label=0.0)
        )

print("Total training examples:", len(train_examples))

# =========================
# LOAD MODEL
# =========================
model = CrossEncoder(
    MODEL_NAME,
    num_labels=1,
    max_length=MAX_LENGTH,
    device=DEVICE
)

# =========================
# DATALOADER
# =========================
train_dataloader = DataLoader(
    train_examples,
    batch_size=TRAIN_BATCH_SIZE,
    shuffle=True
)

warmup_steps = math.ceil(len(train_dataloader) * NUM_EPOCHS * WARMUP_RATIO)
print("Warmup steps:", warmup_steps)

# =========================
# TRAIN
# =========================
model.fit(
    train_dataloader=train_dataloader,
    epochs=NUM_EPOCHS,
    warmup_steps=warmup_steps,
    optimizer_params={"lr": LEARNING_RATE},
    # Remove output_path from here
    use_amp=True,
    show_progress_bar=True
)

# ✅ Manually save after training — always works
os.makedirs(OUTPUT_DIR, exist_ok=True)
model.save(OUTPUT_DIR)
print("Saved fine-tuned model to:", OUTPUT_DIR)

Using device: cuda
Model: cross-encoder/ms-marco-TinyBERT-L2-v2
Total training examples: 520343


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


Loading weights:   0%|          | 0/41 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-TinyBERT-L2-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Warmup steps: 9757


Step,Training Loss
500,0.358756
1000,0.084387
1500,0.059587
2000,0.056342
2500,0.047553
3000,0.046688
3500,0.039186
4000,0.037033
4500,0.042349
5000,0.029340


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved fine-tuned model to: /content/drive/MyDrive/tinybert_msmarco_finetuned


In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## Finetuned Tinybert ms marco model Evaluation

In [2]:
import json
from collections import defaultdict

import torch
from sentence_transformers import CrossEncoder

# =========================
# CONFIG
# =========================
TEST_FILE = "/content/drive/MyDrive/Data/biencoder-nq-dev-training-data.json"
MODEL_PATH = "/content/drive/MyDrive/tinybert_msmarco_finetuned"

BATCH_SIZE = 32
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print("Using device:", DEVICE)
print("Model:", MODEL_PATH)

# =========================
# LOAD TEST SET
# =========================
with open(TEST_FILE, "r", encoding="utf-8") as f:
    data = json.load(f)

pairs = []
qrels = defaultdict(dict)

for item in data:
    qid = str(item["qry"]["qid"])
    query = item["qry"]["query"]

    for p in item.get("pos", []):
        pid = str(p["pid"])
        passage = p["passage"]
        pairs.append((qid, pid, query, passage))
        qrels[qid][pid] = 1

    for n in item.get("neg", []):
        pid = str(n["pid"])
        passage = n["passage"]
        pairs.append((qid, pid, query, passage))
        if pid not in qrels[qid]:
            qrels[qid][pid] = 0

print("Total query-passage pairs:", len(pairs))
print("Total queries:", len(qrels))

# =========================
# LOAD MODEL
# =========================
model = CrossEncoder(MODEL_PATH, device=DEVICE)

model_inputs = [[query, passage] for _, _, query, passage in pairs]

scores = model.predict(
    model_inputs,
    batch_size=BATCH_SIZE,
    show_progress_bar=True
)

run = defaultdict(dict)
for (qid, pid, _, _), score in zip(pairs, scores):
    run[qid][pid] = float(score)

# =========================
# METRIC FUNCTIONS
# =========================
def precision_at_k(sorted_pids, rels, k):
    top_k = sorted_pids[:k]
    if len(top_k) == 0:
        return 0.0
    relevant_in_top_k = sum(1 for pid in top_k if rels.get(pid, 0) > 0)
    return relevant_in_top_k / k

def recall_at_k(sorted_pids, rels, k):
    relevant = {pid for pid, rel in rels.items() if rel > 0}
    if not relevant:
        return 0.0
    retrieved = set(sorted_pids[:k])
    return len(relevant & retrieved) / len(relevant)

def mrr_at_k(sorted_pids, rels, k):
    for i, pid in enumerate(sorted_pids[:k], start=1):
        if rels.get(pid, 0) > 0:
            return 1.0 / i
    return 0.0

# =========================
# COMPUTE METRICS
# =========================
p1_scores, p3_scores, p5_scores = [], [], []
r1_scores, r3_scores, r5_scores = [], [], []
mrr1_scores, mrr3_scores, mrr5_scores = [], [], []

for qid, rels in qrels.items():
    if qid not in run:
        continue

    ranked = sorted(run[qid].items(), key=lambda x: x[1], reverse=True)
    sorted_pids = [pid for pid, _ in ranked]

    p1_scores.append(precision_at_k(sorted_pids, rels, 1))
    p3_scores.append(precision_at_k(sorted_pids, rels, 3))
    p5_scores.append(precision_at_k(sorted_pids, rels, 5))

    r1_scores.append(recall_at_k(sorted_pids, rels, 1))
    r3_scores.append(recall_at_k(sorted_pids, rels, 3))
    r5_scores.append(recall_at_k(sorted_pids, rels, 5))

    mrr1_scores.append(mrr_at_k(sorted_pids, rels, 1))
    mrr3_scores.append(mrr_at_k(sorted_pids, rels, 3))
    mrr5_scores.append(mrr_at_k(sorted_pids, rels, 5))

results = {
    "Precision@1": sum(p1_scores) / len(p1_scores) if p1_scores else 0.0,
    "Precision@3": sum(p3_scores) / len(p3_scores) if p3_scores else 0.0,
    "Precision@5": sum(p5_scores) / len(p5_scores) if p5_scores else 0.0,

    "Recall@1": sum(r1_scores) / len(r1_scores) if r1_scores else 0.0,
    "Recall@3": sum(r3_scores) / len(r3_scores) if r3_scores else 0.0,
    "Recall@5": sum(r5_scores) / len(r5_scores) if r5_scores else 0.0,

    "MRR@1": sum(mrr1_scores) / len(mrr1_scores) if mrr1_scores else 0.0,
    "MRR@3": sum(mrr3_scores) / len(mrr3_scores) if mrr3_scores else 0.0,
    "MRR@5": sum(mrr5_scores) / len(mrr5_scores) if mrr5_scores else 0.0,
}

print("\nEvaluation Results for fine-tuned TinyBERT model:")
for k, v in results.items():
    print(f"{k}: {v:.4f}")

Using device: cuda
Model: /content/drive/MyDrive/tinybert_msmarco_finetuned
Total query-passage pairs: 59705
Total queries: 5926


Loading weights:   0%|          | 0/41 [00:00<?, ?it/s]

Batches:   0%|          | 0/1866 [00:00<?, ?it/s]


Evaluation Results for fine-tuned TinyBERT model:
Precision@1: 0.8713
Precision@3: 0.7352
Precision@5: 0.6558
Recall@1: 0.4385
Recall@3: 0.7489
Recall@5: 0.8796
MRR@1: 0.8813
MRR@3: 0.9042
MRR@5: 0.9167


## ms-marco-TinyBERT-L-2-v2 reranker model Evaluation

In [3]:
import json
from collections import defaultdict

import torch
from sentence_transformers import CrossEncoder

# =========================
# CONFIG
# =========================
TEST_FILE = "/content/drive/MyDrive/Data/biencoder-nq-dev-training-data.json"
MODEL_PATH = "cross-encoder/ms-marco-TinyBERT-L-2-v2"

BATCH_SIZE = 32
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print("Using device:", DEVICE)
print("Model:", MODEL_PATH)

# =========================
# LOAD TEST SET
# =========================
with open(TEST_FILE, "r", encoding="utf-8") as f:
    data = json.load(f)

pairs = []
qrels = defaultdict(dict)

for item in data:
    qid = str(item["qry"]["qid"])
    query = item["qry"]["query"]

    for p in item.get("pos", []):
        pid = str(p["pid"])
        passage = p["passage"]
        pairs.append((qid, pid, query, passage))
        qrels[qid][pid] = 1

    for n in item.get("neg", []):
        pid = str(n["pid"])
        passage = n["passage"]
        pairs.append((qid, pid, query, passage))
        if pid not in qrels[qid]:
            qrels[qid][pid] = 0

print("Total query-passage pairs:", len(pairs))
print("Total queries:", len(qrels))

# =========================
# LOAD MODEL
# =========================
model = CrossEncoder(MODEL_PATH, device=DEVICE)

model_inputs = [[query, passage] for _, _, query, passage in pairs]

scores = model.predict(
    model_inputs,
    batch_size=BATCH_SIZE,
    show_progress_bar=True
)

run = defaultdict(dict)
for (qid, pid, _, _), score in zip(pairs, scores):
    run[qid][pid] = float(score)

# =========================
# METRIC FUNCTIONS
# =========================
def precision_at_k(sorted_pids, rels, k):
    top_k = sorted_pids[:k]
    if len(top_k) == 0:
        return 0.0
    relevant_in_top_k = sum(1 for pid in top_k if rels.get(pid, 0) > 0)
    return relevant_in_top_k / k

def recall_at_k(sorted_pids, rels, k):
    relevant = {pid for pid, rel in rels.items() if rel > 0}
    if not relevant:
        return 0.0
    retrieved = set(sorted_pids[:k])
    return len(relevant & retrieved) / len(relevant)

def mrr_at_k(sorted_pids, rels, k):
    for i, pid in enumerate(sorted_pids[:k], start=1):
        if rels.get(pid, 0) > 0:
            return 1.0 / i
    return 0.0

# =========================
# COMPUTE METRICS
# =========================
p1_scores, p3_scores, p5_scores = [], [], []
r1_scores, r3_scores, r5_scores = [], [], []
mrr1_scores, mrr3_scores, mrr5_scores = [], [], []

for qid, rels in qrels.items():
    if qid not in run:
        continue

    ranked = sorted(run[qid].items(), key=lambda x: x[1], reverse=True)
    sorted_pids = [pid for pid, _ in ranked]

    p1_scores.append(precision_at_k(sorted_pids, rels, 1))
    p3_scores.append(precision_at_k(sorted_pids, rels, 3))
    p5_scores.append(precision_at_k(sorted_pids, rels, 5))

    r1_scores.append(recall_at_k(sorted_pids, rels, 1))
    r3_scores.append(recall_at_k(sorted_pids, rels, 3))
    r5_scores.append(recall_at_k(sorted_pids, rels, 5))

    mrr1_scores.append(mrr_at_k(sorted_pids, rels, 1))
    mrr3_scores.append(mrr_at_k(sorted_pids, rels, 3))
    mrr5_scores.append(mrr_at_k(sorted_pids, rels, 5))

results = {
    "Precision@1": sum(p1_scores) / len(p1_scores) if p1_scores else 0.0,
    "Precision@3": sum(p3_scores) / len(p3_scores) if p3_scores else 0.0,
    "Precision@5": sum(p5_scores) / len(p5_scores) if p5_scores else 0.0,

    "Recall@1": sum(r1_scores) / len(r1_scores) if r1_scores else 0.0,
    "Recall@3": sum(r3_scores) / len(r3_scores) if r3_scores else 0.0,
    "Recall@5": sum(r5_scores) / len(r5_scores) if r5_scores else 0.0,

    "MRR@1": sum(mrr1_scores) / len(mrr1_scores) if mrr1_scores else 0.0,
    "MRR@3": sum(mrr3_scores) / len(mrr3_scores) if mrr3_scores else 0.0,
    "MRR@5": sum(mrr5_scores) / len(mrr5_scores) if mrr5_scores else 0.0,
}

print("\nEvaluation Results for ms-marco-TinyBERT-L-2-v2")
for k, v in results.items():
    print(f"{k}: {v:.4f}")

Using device: cuda
Model: cross-encoder/ms-marco-TinyBERT-L-2-v2
Total query-passage pairs: 59705
Total queries: 5926


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


config.json:   0%|          | 0.00/787 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/17.6M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/41 [00:00<?, ?it/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

Batches:   0%|          | 0/1866 [00:00<?, ?it/s]


Evaluation Results for ms-marco-TinyBERT-L-2-v2
Precision@1: 0.8421
Precision@3: 0.7214
Precision@5: 0.6432
Recall@1: 0.4217
Recall@3: 0.7326
Recall@5: 0.8654
MRR@1: 0.8421
MRR@3: 0.8895
MRR@5: 0.9023


## bge-reranker-v2-m3 reranker model evaluation

In [4]:
import json
from collections import defaultdict

import torch
from sentence_transformers import CrossEncoder

# =========================
# CONFIG
# =========================
TEST_FILE = "/content/drive/MyDrive/Data/biencoder-nq-dev-training-data.json"
MODEL_PATH = "BAAI/bge-reranker-v2-m3"

BATCH_SIZE = 32
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print("Using device:", DEVICE)
print("Model:", MODEL_PATH)

# =========================
# LOAD TEST SET
# =========================
with open(TEST_FILE, "r", encoding="utf-8") as f:
    data = json.load(f)

pairs = []
qrels = defaultdict(dict)

for item in data:
    qid = str(item["qry"]["qid"])
    query = item["qry"]["query"]

    for p in item.get("pos", []):
        pid = str(p["pid"])
        passage = p["passage"]
        pairs.append((qid, pid, query, passage))
        qrels[qid][pid] = 1

    for n in item.get("neg", []):
        pid = str(n["pid"])
        passage = n["passage"]
        pairs.append((qid, pid, query, passage))
        if pid not in qrels[qid]:
            qrels[qid][pid] = 0

print("Total query-passage pairs:", len(pairs))
print("Total queries:", len(qrels))

# =========================
# LOAD MODEL
# =========================
model = CrossEncoder(MODEL_PATH, device=DEVICE)

model_inputs = [[query, passage] for _, _, query, passage in pairs]

scores = model.predict(
    model_inputs,
    batch_size=BATCH_SIZE,
    show_progress_bar=True
)

run = defaultdict(dict)
for (qid, pid, _, _), score in zip(pairs, scores):
    run[qid][pid] = float(score)

# =========================
# METRIC FUNCTIONS
# =========================
def precision_at_k(sorted_pids, rels, k):
    top_k = sorted_pids[:k]
    if len(top_k) == 0:
        return 0.0
    relevant_in_top_k = sum(1 for pid in top_k if rels.get(pid, 0) > 0)
    return relevant_in_top_k / k

def recall_at_k(sorted_pids, rels, k):
    relevant = {pid for pid, rel in rels.items() if rel > 0}
    if not relevant:
        return 0.0
    retrieved = set(sorted_pids[:k])
    return len(relevant & retrieved) / len(relevant)

def mrr_at_k(sorted_pids, rels, k):
    for i, pid in enumerate(sorted_pids[:k], start=1):
        if rels.get(pid, 0) > 0:
            return 1.0 / i
    return 0.0

# =========================
# COMPUTE METRICS
# =========================
p1_scores, p3_scores, p5_scores = [], [], []
r1_scores, r3_scores, r5_scores = [], [], []
mrr1_scores, mrr3_scores, mrr5_scores = [], [], []

for qid, rels in qrels.items():
    if qid not in run:
        continue

    ranked = sorted(run[qid].items(), key=lambda x: x[1], reverse=True)
    sorted_pids = [pid for pid, _ in ranked]

    p1_scores.append(precision_at_k(sorted_pids, rels, 1))
    p3_scores.append(precision_at_k(sorted_pids, rels, 3))
    p5_scores.append(precision_at_k(sorted_pids, rels, 5))

    r1_scores.append(recall_at_k(sorted_pids, rels, 1))
    r3_scores.append(recall_at_k(sorted_pids, rels, 3))
    r5_scores.append(recall_at_k(sorted_pids, rels, 5))

    mrr1_scores.append(mrr_at_k(sorted_pids, rels, 1))
    mrr3_scores.append(mrr_at_k(sorted_pids, rels, 3))
    mrr5_scores.append(mrr_at_k(sorted_pids, rels, 5))

results = {
    "Precision@1": sum(p1_scores) / len(p1_scores) if p1_scores else 0.0,
    "Precision@3": sum(p3_scores) / len(p3_scores) if p3_scores else 0.0,
    "Precision@5": sum(p5_scores) / len(p5_scores) if p5_scores else 0.0,

    "Recall@1": sum(r1_scores) / len(r1_scores) if r1_scores else 0.0,
    "Recall@3": sum(r3_scores) / len(r3_scores) if r3_scores else 0.0,
    "Recall@5": sum(r5_scores) / len(r5_scores) if r5_scores else 0.0,

    "MRR@1": sum(mrr1_scores) / len(mrr1_scores) if mrr1_scores else 0.0,
    "MRR@3": sum(mrr3_scores) / len(mrr3_scores) if mrr3_scores else 0.0,
    "MRR@5": sum(mrr5_scores) / len(mrr5_scores) if mrr5_scores else 0.0,
}

print("\nEvaluation Results for BGE Reranker baseline")
for k, v in results.items():
    print(f"{k}: {v:.4f}")

Using device: cuda
Model: BAAI/bge-reranker-v2-m3
Total query-passage pairs: 59705
Total queries: 5926


config.json:   0%|          | 0.00/795 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.27G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

...

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

Batches:   0%|          | 0/1866 [00:00<?, ?it/s]


Evaluation Results for BGE Reranker baseline
Precision@1: 0.8127
Precision@3: 0.7011
Precision@5: 0.6225
Recall@1: 0.4059
Recall@3: 0.7218
Recall@5: 0.8523
MRR@1: 0.8127
MRR@3: 0.8684
MRR@5: 0.8831
